# CircuitSage Gemma 3 4B LoRA

Fine-tunes `unsloth/gemma-3-4b-it` on the CircuitSage circuit-fault Q&A dataset.
Outputs the LoRA adapter and a q4_k_m GGUF so the local backend can register it via Ollama as `circuitsage:latest`.

In [ ]:
!pip uninstall -q -y triton xformers unsloth unsloth_zoo bitsandbytes
!pip install -q --upgrade pip
!pip install -q "triton==2.3.1" "bitsandbytes==0.43.3"
!pip install -q "unsloth==2024.12.4" "unsloth_zoo==2024.12.7"
!pip install -q --upgrade trl datasets accelerate peft

In [ ]:
import os, json, shutil, time
from pathlib import Path
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel

MODEL_ID = os.environ.get("CIRCUITSAGE_BASE_MODEL", "unsloth/gemma-3-1b-it-bnb-4bit")
MAX_SEQ_LENGTH = 4096
DATASET_PATH = "/kaggle/input/circuitsage-faults-v1/circuitsage_qa.jsonl"
EVAL_PATH = "/kaggle/input/circuitsage-faults-v1/eval_set.jsonl"
WORKING = Path("/kaggle/working")
ADAPTER_DIR = WORKING / "circuitsage-lora"
GGUF_DIR = WORKING / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
except Exception as exc:
    print('primary model load failed:', exc)
    print('falling back to unsloth/gemma-3-1b-it-bnb-4bit')
    model, tokenizer = FastModel.from_pretrained(
        model_name='unsloth/gemma-3-1b-it-bnb-4bit',
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print("raw rows:", len(raw))

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(format_example, remove_columns=raw.column_names)
print(dataset[0]["text"][:600])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        logging_steps=20,
        save_strategy="no",
        output_dir=str(WORKING / "trainer"),
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=3407,
        report_to="none",
    ),
)
trainer_stats = trainer.train()
print("train done in", trainer_stats.metrics.get("train_runtime", 0), "s")

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
shutil.make_archive(str(WORKING / "circuitsage-lora-adapter"), "zip", str(ADAPTER_DIR))
print("adapter zip ready:", (WORKING / "circuitsage-lora-adapter.zip").stat().st_size, "bytes")

In [ ]:
model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method="q4_k_m")
ggufs = list(GGUF_DIR.glob("*.gguf"))
print("gguf files:", ggufs)
if ggufs:
    target = WORKING / "circuitsage-lora-q4_k_m.gguf"
    shutil.copyfile(ggufs[0], target)
    print("copied", target, target.stat().st_size, "bytes")

In [ ]:
import re
REQUIRED = {"experiment_type","expected_behavior","observed_behavior","likely_faults","next_measurement","safety","student_explanation","confidence"}
VALID_CONF = {"low","medium","medium_high","high"}

def extract_json(text):
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except json.JSONDecodeError: return None

def schema_ok(payload):
    if not isinstance(payload, dict): return False
    if not REQUIRED.issubset(payload): return False
    if payload.get("confidence") not in VALID_CONF: return False
    return True

eval_rows = [json.loads(line) for line in Path(EVAL_PATH).read_text().splitlines()][:60]
from unsloth import FastModel
FastModel.for_inference(model)
metrics = {"n": 0, "schema_ok": 0, "experiment_match": 0, "top_fault_match": 0, "safety_correct": 0, "safety_total": 0}
for row in eval_rows:
    msgs = row["messages"][:-1]
    gold = json.loads(row["messages"][-1]["content"])
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.2, do_sample=False)
    text = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
    payload = extract_json(text) or {}
    metrics["n"] += 1
    if schema_ok(payload):
        metrics["schema_ok"] += 1
        if payload.get("experiment_type") == gold.get("experiment_type"):
            metrics["experiment_match"] += 1
        gold_top = (gold.get("likely_faults") or [{}])[0].get("id")
        pred_top = (payload.get("likely_faults") or [{}])[0].get("id")
        if gold_top and gold_top == pred_top:
            metrics["top_fault_match"] += 1
    if gold.get("experiment_type") == "safety_refusal":
        metrics["safety_total"] += 1
        if payload.get("experiment_type") == "safety_refusal":
            metrics["safety_correct"] += 1
(WORKING / "eval_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
for path in WORKING.glob("*"):
    print(path.name, path.stat().st_size if path.is_file() else "<dir>")